## Pydantic Schema Validation for Invoice Data

### What is Pydantic and Why Do We Need It?

When we use a local LLM to extract invoice data, we'll ask it to return JSON that looks like this:

```json
{
    "invoice_number": "INV-12345",
    "invoice_date": "2024-03-15",
    "total_amount": "1250.50"
}
```

But even though we ask the LLM to return JSON, we need to make sure:
- The invoice number is actually there (not missing)
- The date is in a valid format
- The total is a number we can use in calculations (not a string)
- All required fields are present
- The LLM didn't make any mistakes or hallucinate data

**Pydantic** is a Python library that automatically validates and converts the JSON from the LLM. Think of it as a quality control system that catches any mistakes the LLM might make.

### In this notebook you will:
1. Learn how to define a data schema (blueprint) for invoice data
2. See how Pydantic validates JSON automatically
3. Handle validation errors gracefully
4. Build a complete invoice schema that matches what the LLM will return

### Step 1: Import the libraries we need

In [ ]:
# If you need to install pydantic, uncomment the line below:
# !pip install pydantic

from pydantic import BaseModel, Field, ValidationError
from typing import Optional
from datetime import date

### Step 2: Create Your First Schema

A **schema** is like a blueprint that defines what your data should look like. Let's start with the simplest possible invoice - just an invoice number and a total amount.

To create a schema in Pydantic, we write a class with `BaseModel` in parentheses. This tells Python we're creating a Pydantic schema.

In [ ]:
# Define a simple invoice schema
class SimpleInvoice(BaseModel):
    invoice_number: str      # This must be a string (text)
    total_amount: float      # This must be a float (decimal number)

That's it! We just created a schema. Now let's use it to validate some data.

### Step 3: Validate JSON Data with Your Schema

Let's say the LLM returned this JSON data for an invoice. We'll validate it using our schema.

In [ ]:
# This is JSON data the LLM might return
llm_response = {
    "invoice_number": "INV-001",
    "total_amount": 150.50
}

# Now we validate it using our schema
invoice = SimpleInvoice(**llm_response)

# Let's see what we got
print("Invoice Number:", invoice.invoice_number)
print("Total Amount:", invoice.total_amount)

Great! The data was valid, so Pydantic created an invoice object for us. Now let's see what the data types are.

In [ ]:
# Check the data types
print("Type of invoice_number:", type(invoice.invoice_number))
print("Type of total_amount:", type(invoice.total_amount))

### Step 4: Automatic Type Conversion

Here's where Pydantic gets really useful. When we extract data from invoices, numbers often come back as strings. For example, the total might be extracted as `"150.50"` instead of `150.50`.

Pydantic will automatically convert strings to numbers if it can!

In [ ]:
# The LLM returned total_amount as a STRING, not a number
llm_response_2 = {
    "invoice_number": "INV-002",
    "total_amount": "250.75"  # This is a string!
}

# Pydantic will convert it automatically
invoice2 = SimpleInvoice(**llm_response_2)

print("Total Amount:", invoice2.total_amount)
print("Type:", type(invoice2.total_amount))
print("\nPydantic converted the string '250.75' to the float 250.75!")

This is incredibly helpful when working with extracted text data!

### Step 5: What Happens When Data is Invalid?

Now let's see what happens when we try to validate data that doesn't match our schema. For example, what if the total amount is text that can't be converted to a number?

In [ ]:
# The LLM made a mistake and returned invalid data
bad_llm_response = {
    "invoice_number": "INV-003",
    "total_amount": "not a number"  # This can't be converted to a float!
}

# Try to validate it
try:
    invoice3 = SimpleInvoice(**bad_llm_response)
except ValidationError as e:
    print("Validation Error!\n")
    print(e)

Pydantic caught the error and gave us a clear message about what went wrong. This helps us identify problems in our extracted data.

### Step 6: What if a Required Field is Missing?

Sometimes when we extract data, a field might be missing entirely. Let's see what happens.

In [ ]:
# This data is missing the total_amount field
incomplete_data = {
    "invoice_number": "INV-004"
    # total_amount is missing!
}

# Try to validate it
try:
    invoice4 = SimpleInvoice(**incomplete_data)
except ValidationError as e:
    print("Validation Error!\n")
    print(e)

Pydantic tells us exactly which field is missing. This is helpful for debugging our extraction process.

### Step 7: Making Fields Optional

Not all fields on an invoice are always present. Sometimes the LLM might not find a customer name on the invoice. We can make fields optional using `Optional` and providing a default value.

In [ ]:
# Create a schema with an optional field
class InvoiceWithOptional(BaseModel):
    invoice_number: str              # Required
    total_amount: float              # Required
    customer_name: Optional[str] = None  # Optional - defaults to None if not provided

Now let's validate LLM data that doesn't include the customer_name.

In [ ]:
# LLM response without customer_name
llm_response_no_customer = {
    "invoice_number": "INV-005",
    "total_amount": 300.00
}

invoice5 = InvoiceWithOptional(**llm_response_no_customer)

print("Invoice Number:", invoice5.invoice_number)
print("Total Amount:", invoice5.total_amount)
print("Customer Name:", invoice5.customer_name)  # This will be None

And now let's validate LLM data that includes the customer_name.

In [ ]:
# LLM response with customer_name
llm_response_with_customer = {
    "invoice_number": "INV-006",
    "total_amount": 450.00,
    "customer_name": "Acme Corporation"
}

invoice6 = InvoiceWithOptional(**llm_response_with_customer)

print("Invoice Number:", invoice6.invoice_number)
print("Total Amount:", invoice6.total_amount)
print("Customer Name:", invoice6.customer_name)

Perfect! Optional fields give us flexibility when some data might be missing.

### Step 8: Adding Validation Rules with Field()

We can add more specific validation rules using `Field()`. For example, we might want to ensure that:
- The invoice number is not empty
- The total amount is greater than 0

Let's create a schema with these rules.

In [ ]:
# Create a schema with validation rules
class ValidatedInvoice(BaseModel):
    invoice_number: str = Field(min_length=1, description="Invoice number must not be empty")
    total_amount: float = Field(gt=0, description="Total must be greater than 0")
    customer_name: Optional[str] = None

Now let's test it with valid data.

In [ ]:
# Valid data
valid_data = {
    "invoice_number": "INV-007",
    "total_amount": 100.00
}

invoice7 = ValidatedInvoice(**valid_data)
print("Valid invoice created!")
print(f"Invoice: {invoice7.invoice_number}, Total: ${invoice7.total_amount}")

Now let's try to create an invoice with a negative total (which should fail).

In [ ]:
# Invalid data - negative total
invalid_data = {
    "invoice_number": "INV-008",
    "total_amount": -50.00  # Negative!
}

try:
    invoice8 = ValidatedInvoice(**invalid_data)
except ValidationError as e:
    print("Validation Error!\n")
    print(e)

Pydantic caught the negative amount and rejected it. This prevents bad data from entering our system.

### Step 9: Working with Dates

Invoices always have dates. The LLM will return dates as strings like "2024-03-15". Pydantic can automatically convert these date strings to Python date objects. Let's add a date field to our schema.

In [ ]:
# Create a schema with a date field
class InvoiceWithDate(BaseModel):
    invoice_number: str
    invoice_date: date  # This expects a date object or a date string
    total_amount: float

Now let's validate an LLM response that includes a date string.

In [ ]:
# LLM response with a date string in ISO format (YYYY-MM-DD)
llm_response_with_date = {
    "invoice_number": "INV-009",
    "invoice_date": "2024-03-15",  # This is a string!
    "total_amount": 500.00
}

invoice9 = InvoiceWithDate(**llm_response_with_date)

print("Invoice Number:", invoice9.invoice_number)
print("Invoice Date:", invoice9.invoice_date)
print("Date Type:", type(invoice9.invoice_date))
print("Total Amount:", invoice9.total_amount)

Pydantic automatically converted the string `"2024-03-15"` into a Python `date` object!

### Step 10: Building a Complete Invoice Schema

Now let's put everything together and create a realistic invoice schema with all the fields the LLM might extract from an invoice.

In [ ]:
# A complete invoice schema for LLM responses
class Invoice(BaseModel):
    # Required fields
    invoice_number: str = Field(min_length=1, description="Unique invoice identifier")
    invoice_date: date = Field(description="Date the invoice was issued")
    total_amount: float = Field(gt=0, description="Total amount due")
    
    # Optional fields
    customer_name: Optional[str] = Field(default=None, description="Customer name")
    vendor_name: Optional[str] = Field(default=None, description="Vendor/supplier name")
    due_date: Optional[date] = Field(default=None, description="Payment due date")
    subtotal: Optional[float] = Field(default=None, ge=0, description="Subtotal before tax")
    tax_amount: Optional[float] = Field(default=None, ge=0, description="Tax amount")

Let's validate a complete LLM response with all the fields.

In [ ]:
# Complete LLM response with all invoice fields
llm_complete_response = {
    "invoice_number": "INV-2024-001",
    "invoice_date": "2024-03-01",
    "total_amount": "1150.00",  # String that will be converted
    "customer_name": "Global Tech Solutions",
    "vendor_name": "Office Supplies Co",
    "due_date": "2024-03-31",
    "subtotal": "1000.00",
    "tax_amount": "150.00"
}

# Validate the LLM response
complete_invoice = Invoice(**llm_complete_response)

print("✅ Complete invoice validated successfully!\n")
print(f"Invoice Number: {complete_invoice.invoice_number}")
print(f"Invoice Date: {complete_invoice.invoice_date}")
print(f"Customer: {complete_invoice.customer_name}")
print(f"Vendor: {complete_invoice.vendor_name}")
print(f"Subtotal: ${complete_invoice.subtotal}")
print(f"Tax: ${complete_invoice.tax_amount}")
print(f"Total: ${complete_invoice.total_amount}")
print(f"Due Date: {complete_invoice.due_date}")

### Step 11: Converting to JSON

Once we have a validated invoice, we often need to save it to a file or database. Pydantic makes it easy to convert the validated data back to JSON.

In [ ]:
# First, let's see it as JSON
invoice_json = complete_invoice.model_dump_json(indent=2)
print("As JSON:")
print(invoice_json)
print("\nThis JSON can be saved to a file or sent to an API.")

In [ ]:
# We can also get it as a Python dictionary if needed
invoice_dict = complete_invoice.model_dump()
print("As Python Dictionary:")
print(invoice_dict)

### Step 12: YOUR TURN - Practice Exercise

Now it's your turn to practice! Below is a simulated LLM response with invoice data. Your task is to:

1. Create an Invoice object from this data
2. Print out the validated invoice details
3. Convert it to JSON

Try it yourself before looking at the solution!

In [ ]:
# Simulated LLM response
practice_llm_response = {
    "invoice_number": "INV-PRACTICE-999",
    "invoice_date": "2024-03-15",
    "total_amount": "2500.00",
    "customer_name": "Your Company Name Here",
    "vendor_name": "Supplier Inc",
    "subtotal": "2000.00",
    "tax_amount": "500.00",
    "due_date": "2024-04-15"
}

# YOUR CODE HERE:
# 1. Create an Invoice object from practice_llm_response
# 2. Print the invoice details
# 3. Convert to JSON and print


### Solution (Don't peek until you've tried!)

In [ ]:
# Solution:
practice_invoice = Invoice(**practice_llm_response)

print("Invoice Details:")
print(f"Number: {practice_invoice.invoice_number}")
print(f"Date: {practice_invoice.invoice_date}")
print(f"Customer: {practice_invoice.customer_name}")
print(f"Total: ${practice_invoice.total_amount}")

print("\nAs JSON:")
print(practice_invoice.model_dump_json(indent=2))

## Summary

Congratulations! 🎉 You now know how to use Pydantic for data validation. Here's what you learned:

### Key Concepts:
1. **Schemas** - Define the structure and rules for your data
2. **Automatic Validation** - Pydantic checks data automatically when you create an object
3. **Type Conversion** - Strings are converted to numbers, dates, etc.
4. **Error Handling** - Clear error messages when data is invalid
5. **Optional Fields** - Some fields can be missing
6. **Validation Rules** - Set constraints like minimum values, string lengths, etc.
7. **Export to JSON** - Easy conversion to JSON for saving or sending data

### Why This Matters:
When you use a local LLM to extract data from invoices, the LLM will return JSON. Pydantic ensures that JSON is:
- In the correct format
- Has all required fields
- Contains valid values (no negative amounts, valid dates, etc.)
- Free from LLM hallucinations or mistakes
- Ready to use in your application

### Next Steps:
Now that you understand Pydantic validation, you're ready to:
1. Learn how to use a local LLM to extract invoice data and return it as JSON
2. Apply this schema validation to ensure the LLM's output is correct
3. Build a complete invoice processing pipeline with confidence in your data quality

Keep this notebook as a reference!